# Larger Qwen2.5 on the Test Set (Colab, self-contained)

Runs `qwen2.5:32b` or `qwen2.5:72b` (generation and candidate selection) on the frozen
`data/splits/test_items.csv`, with the same prompts, shuffle seed, scorer and default
decoding as the `qwen2.5:7b` run. Only the model size changes.

Same pattern as `run_test_eval_colab.ipynb`: no repo clone and no `pip install -e .`;
the few small source files are fetched from GitHub and imported as loose modules.
Ollama is installed inside the Colab session.

**Runtime:** set `MODEL` below, then `Runtime > Change runtime type`.
- `qwen2.5:32b` (~20 GB): L4 (24 GB) is tight; A100 40 GB is comfortable.
- `qwen2.5:72b` (~47 GB): needs an 80 GB GPU (A100 80 GB or H100).

Outputs are saved after each mode and downloaded. Re-running skips a mode already saved.

## 1. Fetch source files

In [ ]:
!pip install -q requests tqdm

In [ ]:
import urllib.request, sys, types, importlib.util

REPO_RAW = 'https://raw.githubusercontent.com/BenCarmel123/hebrew-acronym-disambiguation/main'

# dependency order: csv_io -> pairs -> eval -> qwen.eval
FILES = {
    'hebrew_acronyms.data_processing.common.csv_io': 'src/hebrew_acronyms/data_processing/common/csv_io.py',
    'hebrew_acronyms.models.common.pairs': 'src/hebrew_acronyms/models/common/pairs.py',
    'hebrew_acronyms.models.common.eval': 'src/hebrew_acronyms/models/common/eval.py',
    'hebrew_acronyms.models.qwen.eval': 'src/hebrew_acronyms/models/qwen/eval.py',
}

# empty package stubs so the dotted imports resolve without installing the package
pkgs = {'.'.join(d.split('.')[:i]) for d in FILES for i in range(1, len(d.split('.')))}
for pkg in sorted(pkgs, key=len):
    mod = types.ModuleType(pkg)
    mod.__path__ = []
    sys.modules[pkg] = mod

def fetch_module(dotted_name, repo_path):
    local_path = dotted_name.replace('.', '_') + '.py'
    urllib.request.urlretrieve(f'{REPO_RAW}/{repo_path}', local_path)
    spec = importlib.util.spec_from_file_location(dotted_name, local_path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[dotted_name] = mod
    spec.loader.exec_module(mod)

for dotted, path in FILES.items():
    fetch_module(dotted, path)

urllib.request.urlretrieve(f'{REPO_RAW}/data/splits/test_items.csv', 'test_items.csv')
print('modules and test_items.csv fetched')

## 2. Install and start Ollama

`zstd` is needed by the Ollama installer. If Colab asks for a runtime restart, restart and
re-run from the top; nothing here depends on state surviving a restart.

In [ ]:
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time
ollama_proc = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)
print('ollama server started, pid', ollama_proc.pid)

## 3. Choose and pull the model

In [ ]:
MODEL = 'qwen2.5:32b'   # or 'qwen2.5:72b' (needs an 80 GB GPU)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!ollama pull {MODEL}

## 4. Run generation and selection on the test set

In [ ]:
import os
from hebrew_acronyms.models.common.eval import evaluate, write_details_csv
from hebrew_acronyms.models.common.pairs import load_rows
from hebrew_acronyms.models.qwen.eval import ollama_generate, ollama_model_identity

rows = load_rows('test_items.csv')
print('digest:', ollama_model_identity(MODEL)['digest'])

tag = MODEL.replace('qwen2.5:', 'qwen')
os.makedirs(tag, exist_ok=True)
for mode in ('select', 'generate'):
    out = f'{tag}/test_{mode}_details.csv'
    if os.path.exists(out):
        print(mode, 'already saved, skipping')
        continue
    res = evaluate(rows, generate_fn=lambda p: ollama_generate(p, model=MODEL, timeout=600), mode=mode)
    print(f"{MODEL} {mode}: accuracy {res['accuracy']:.3f}  invalid {res['invalid_rate']:.3f}  n {res['n_items']}")
    write_details_csv(out, res['details'])

## 5. Download the results

In [ ]:
!zip -qr {tag}.zip {tag}
from google.colab import files
files.download(f'{tag}.zip')